# Output 3-2 — Public stress sensitivity tables

Excel analogue: **Output 3-2 Stress-public** — one block per indicator,
rows = Baseline + bound tests, columns = projection years.
(**Output 2-2** charts use the same series.)

Covers public **B1 GDP** from Input 6 + Input 7 ResFin (three-way fill).
A* historical / unchanged-PB / lower-growth, public B2–B5 bound tests,
and tailored C* tests are not runners yet.

See `docs/08-stress-dsa.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.load import (
    load_core,
    load_input6_standard,
    load_input7_residual_params,
)

from lic_dsf.stress import run_b1_gdp_public

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/excel-grapher/py-lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [2]:
macro, external, _ext_base, pub_base = load_core(WORKBOOK)
input6 = load_input6_standard(WORKBOOK)
residual = load_input7_residual_params(WORKBOOK)
public_b1 = run_b1_gdp_public(macro, external, input6, residual)

first_proj = macro.inputs.first_projection_year
years = list(range(first_proj, first_proj + 11))
public_b1.scenario_id, years[0], years[-1]

('B1_GDP_pub', 2024, 2034)

## Sensitivity tables (Output 3-2 shape)

Each block: rows = Baseline + B1, columns = years.

In [3]:
def _ratio_row(label: str, series: pd.Series, yrs: list[int]) -> pd.Series:
    return series.reindex(yrs).rename(label)


def _stress_table(baseline: pd.Series, shocked: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(
        [
            _ratio_row("Baseline", baseline, years),
            _ratio_row("B1. Real GDP growth", shocked, years),
        ]
    )


out_3_2 = {
    "PV of debt-to-GDP": _stress_table(
        pub_base.pv_public_debt_to_gdp(),
        public_b1.pv_public_debt_to_gdp(),
    ),
    "PV of debt-to-revenue": _stress_table(
        pub_base.pv_public_debt_to_revenue_grants(),
        public_b1.pv_public_debt_to_revenue_grants(),
    ),
    "Debt service-to-revenue": _stress_table(
        pub_base.debt_service_to_revenue_grants(),
        public_b1.debt_service_to_revenue_grants(),
    ),
}
list(out_3_2)

['PV of debt-to-GDP', 'PV of debt-to-revenue', 'Debt service-to-revenue']

In [4]:
out_3_2["PV of debt-to-GDP"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,67.1667,62.4801,58.6475,55.9502,53.6994,51.4136,49.3587,47.9421,46.9933,46.1724,45.3455
B1. Real GDP growth,67.1667,69.4333,69.8627,68.7502,68.2794,67.4807,66.8920,67.0764,67.8312,68.7001,69.2672


In [5]:
out_3_2["PV of debt-to-revenue"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,356.2547,335.3506,309.1418,293.6842,284.3530,274.2625,266.7039,262.4111,260.0835,257.8026,254.9091
B1. Real GDP growth,356.2547,372.6707,368.2589,360.8714,361.5585,359.9715,361.4430,367.1428,375.4103,383.5852,389.3847


In [6]:
out_3_2["Debt service-to-revenue"]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,126.3839,92.4878,85.3944,82.5395,87.6262,79.1716,80.1363,82.3079,84.4986,86.5869,78.9340
B1. Real GDP growth,126.3839,92.4878,87.7751,88.3005,96.0198,89.9346,93.0394,97.5615,102.5164,107.6876,103.2414
